In [56]:
#라이브러리 로드

import pandas as pd
import numpy as np
import json
import warnings
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
import os
import ast

 
warnings.filterwarnings('ignore')
 
# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False



In [46]:
#데이터 로드 

# 데이터 경로
DATA_DIR = 'C:/project_3/Third-project-sparta/works/doha_Kim/raw_data'
TIERS = ['Platinum', 'Diamond', 'Master', 'GrandMaster', 'Challenger']

# 챔피언 정보 로드
champions_df = pd.read_csv(f'{DATA_DIR}/TFT_Champion_CurrentVersion.csv')
champions = champions_df.copy()
print(f"챔피언 정보: {len(champions_df)}")
 
# 아이템 정보 로드
items_df = pd.read_csv(f'{DATA_DIR}/TFT_Item_CurrentVersion.csv')
items = items_df.copy()
item_dict = dict(zip(items['id'], items['name']))
print(f"아이템 정보: {len(items_df)}")

# 각 티어별 데이터 로드
all_data = {}
for tier in TIERS:
    file_path = f'{DATA_DIR}/TFT_{tier}_MatchData.csv'
    try:
        df_raw = pd.read_csv(file_path) 
        df = df_raw.copy()
        df['tier'] = tier
        all_data[tier] = df
        print(f"{tier} 데이터: {len(df)}")
    except FileNotFoundError:
        print(f"{tier} 데이터 파일 없음")
 
# 전체 데이터 병합
full_data = pd.concat(all_data.values(), ignore_index=True)
print(f"전체 데이터: {len(full_data)} rows, {len(full_data.columns)}컬럼")

챔피언 정보: 52
아이템 정보: 54
Platinum 데이터: 80000
Diamond 데이터: 80000
Master 데이터: 79999
GrandMaster 데이터: 80000
Challenger 데이터: 79999
전체 데이터: 399998 rows, 9컬럼


In [47]:
full_data
full_data.dtypes

gameId                str
gameDuration      float64
level               int64
lastRound           int64
Ranked              int64
ingameDuration    float64
combination           str
champion              str
tier                  str
dtype: object

In [48]:
# 데이터 기본 정보
 
print("컬럼 정보:")
print(full_data.columns.tolist())
print('')
# print("데이터 샘플")
# print(full_data[['gameId', 'tier', 'Ranked', 'level', 'lastRound']].head(2))
print('')
print("결측치 확인:")
print(full_data.isnull().sum())
print('')
print("티어별 rows:")
print(full_data['tier'].value_counts().sort_index())

컬럼 정보:
['gameId', 'gameDuration', 'level', 'lastRound', 'Ranked', 'ingameDuration', 'combination', 'champion', 'tier']


결측치 확인:
gameId            0
gameDuration      0
level             0
lastRound         0
Ranked            0
ingameDuration    0
combination       0
champion          0
tier              0
dtype: int64

티어별 rows:
tier
Challenger     79999
Diamond        80000
GrandMaster    80000
Master         79999
Platinum       80000
Name: count, dtype: int64


In [49]:
game_counts = full_data.groupby('gameId').size()
print(game_counts)

gameId
KR_3890408252    8
KR_3891371111    8
KR_3891442705    8
KR_3891808329    8
KR_3965402291    8
                ..
KR_4402761495    8
KR_4402771106    8
KR_4402800301    8
KR_4402801838    8
KR_4402801990    8
Length: 49981, dtype: int64


In [50]:
# 중복행
duplicates = full_data[full_data.duplicated(keep=False)]

print(f"중복 제거 전: {len(full_data)}")
print(f"중복된 행 개수: {len(duplicates)}")
print()

# 중복행 샘플
print(duplicates.head(20))
print()

# 중복 제거
full_data = full_data.drop_duplicates()
print(f"중복 제거 후: {len(full_data)}")

중복 제거 전: 399998
중복된 행 개수: 80

              gameId  gameDuration  level  lastRound  Ranked  ingameDuration  \
657    KR_4229542485    127.697548      3          4       0      127.697548   
658    KR_4229542485    127.697548      3          4       0      127.697548   
659    KR_4229542485    127.697548      3          4       0      127.697548   
660    KR_4229542485    127.697548      3          4       0      127.697548   
661    KR_4229542485    127.697548      3          4       0      127.697548   
662    KR_4229542485    127.697548      3          4       0      127.697548   
663    KR_4229542485    127.697548      3          4       0      127.697548   
18009  KR_4236285365    452.641998      4         10       0      452.641998   
18010  KR_4236285365    452.641998      4         10       0      452.641998   
18011  KR_4236285365    452.641998      4         10       0      452.641998   
18012  KR_4236285365    452.641998      4         10       0      452.641998   
18013  KR_

In [51]:
game_counts = full_data.groupby('gameId').size()
print(game_counts)

gameId
KR_3890408252    1
KR_3891371111    1
KR_3891442705    1
KR_3891808329    1
KR_3965402291    8
                ..
KR_4402761495    8
KR_4402771106    8
KR_4402800301    8
KR_4402801838    8
KR_4402801990    8
Length: 49981, dtype: int64


In [ ]:
game_counts = full_data.groupby('gameId').size()
invalid_games = game_counts[game_counts != 8].reset_index()
invalid_games.columns = ['gameId', 'count']
# invalid_games.count() # 33개의 게임id

invalid_game_ids = invalid_games['gameId'].tolist()
tier_list = full_data[full_data['gameId'].isin(invalid_game_ids)][['gameId', 'tier']]

# gameId별 tier별 인원수 계산
tier_list = tier_list.groupby(['gameId', 'tier']).size().reset_index(name='tier_count')

# merge 
invalid_games = invalid_games.merge(tier_list, on='gameId')
print(invalid_games)


           gameId  count         tier  tier_count
0   KR_3890408252      1     Platinum           1
1   KR_3891371111      1     Platinum           1
2   KR_3891442705      1     Platinum           1
3   KR_3891808329      1     Platinum           1
4   KR_4229542485      2     Platinum           2
5   KR_4231144753      6     Platinum           6
6   KR_4231208133      3     Platinum           3
7   KR_4236285365      3     Platinum           3
8   KR_4263820118     16       Master           8
9   KR_4263820118     16     Platinum           8
10  KR_4295022760      3       Master           3
11  KR_4303195386      3       Master           3
12  KR_4313697214     16       Master           8
13  KR_4313697214     16     Platinum           8
14  KR_4318806255      7   Challenger           7
15  KR_4320079059     16      Diamond           8
16  KR_4320079059     16     Platinum           8
17  KR_4323594767      2     Platinum           2
18  KR_4335870255      7       Master           7
